<a href="https://colab.research.google.com/github/djqx7/iasnlp_2026/blob/main/llm/speach_text_input.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --no-cache-dir \
  "transformers==4.52.3" \
  "tokenizers==0.21.4" \
  "accelerate>=1.2.0" \
  "huggingface_hub>=0.30.0" \
  "qwen-omni-utils" \
  soundfile librosa pandas scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 80.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 204.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 238.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 101.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 183.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 MB 153.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incom

In [2]:
from google.colab import files

uploaded = files.upload()

Saving audio_files.zip to audio_files (1).zip


In [1]:
  import transformers
import tokenizers
import huggingface_hub

print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

print("Qwen2.5 Omni imports successful")
import zipfile
import os

zip_path = "audio_files.zip"
audio_dir = "audio_files"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(".")

print("Files extracted.")

print("Checking audio folder:")
print(os.listdir(audio_dir)[:10])
print("Total audio files:", len(os.listdir(audio_dir)))

transformers: 4.52.3
tokenizers: 0.21.4
huggingface_hub: 0.36.2
Qwen2.5 Omni imports successful
Files extracted.
Checking audio folder:
['0_1_d1680.wav', '12_0_d66.wav', '0_0_d749.wav', '1_1_d2027.wav', '1_0_d706.wav', '2_0_d1891.wav', '1_0_d762.wav', '0_1_d1817.wav', '1_0_d2539.wav', '12_1_d1292.wav']
Total audio files: 404


In [3]:
import os
import re
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

file_path = "test_dataset_final.csv"
audio_dir = "audio_files"

df = pd.read_csv(file_path)

print("CSV preview:")
print(df.head())
print("Columns:", df.columns)

text_col = "transcript"
audio_col = "file_name"
label_col = "label"

df = df.dropna(subset=[text_col, audio_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Audio column:", audio_col)
print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

missing = []

for f in df[audio_col].astype(str).tolist():
    p = os.path.join(audio_dir, f)
    if not os.path.exists(p):
        missing.append(f)

print("Missing audio files:", len(missing))

if len(missing) > 0:
    print("First missing files:", missing[:10])
    raise FileNotFoundError("Some audio files from CSV are not present in audio_files folder.")

print("\nChecking first 3 audio-text pairs:")
for i in range(min(3, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    label = str(df.loc[i, label_col])

    print("\nSample", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)
    print("True label:", label)

model_name = "Qwen/Qwen2.5-Omni-3B"

print("\nLoading model:", model_name)

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

processor = Qwen2_5OmniProcessor.from_pretrained(model_name)

model.eval()

print("Model loaded successfully.")

def make_prompt(transcript):
    prompt = f"""You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Transcript:
{transcript}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Answer with only the label.

Label:"""

    return prompt

def clean_pred(ans):
    raw = ans
    ans = ans.strip().lower()

    ans = ans.replace("answer:", " ")
    ans = ans.replace("label:", " ")

    ans = re.sub(r"[^a-zA-Z]", " ", ans)
    ans = " ".join(ans.split())

    words = ans.split()

    for lab in labels:
        if ans == lab.lower():
            return lab

    for w in words:
        for lab in labels:
            if w == lab.lower():
                return lab

    for lab in labels:
        if lab.lower() in ans:
            return lab

    print("Could not clean prediction. Raw answer was:", raw)
    return ans

def print_input_debug(i, audio_path, transcript, conversation, text, audios, images, videos, inputs):
    print("\n" + "=" * 80)
    print("DEBUG SAMPLE:", i)
    print("Audio path:", audio_path)
    print("Audio exists:", os.path.exists(audio_path))
    print("Audio size bytes:", os.path.getsize(audio_path))
    print("Transcript:", transcript)

    print("\nConversation content types:")
    for msg in conversation:
        print("Role:", msg["role"])
        for item in msg["content"]:
            print("  type:", item["type"])

    print("\nChat template first 600 characters:")
    print(text[:600])

    print("\nMultimodal objects after process_mm_info:")
    print("Number of audios:", 0 if audios is None else len(audios))
    print("Number of images:", 0 if images is None else len(images))
    print("Number of videos:", 0 if videos is None else len(videos))

    if audios is not None and len(audios) > 0:
        a0 = audios[0]
        print("First audio object type:", type(a0))

        if hasattr(a0, "shape"):
            print("First audio shape:", a0.shape)

        if hasattr(a0, "dtype"):
            print("First audio dtype:", a0.dtype)

    print("\nProcessor input keys:")
    print(list(inputs.keys()))

    print("\nProcessor input tensor details:")
    for k, v in inputs.items():
        if hasattr(v, "shape"):
            print(k, "shape:", tuple(v.shape), "dtype:", v.dtype, "device before move:", v.device)
        else:
            print(k, "type:", type(v))

    if "input_ids" in inputs:
        print("Text input token length:", inputs["input_ids"].shape[1])

    print("=" * 80 + "\n")

def predict(audio_path, transcript, i=None, debug=False):
    prompt = make_prompt(transcript)

    conversation = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory inputs and generating text responses."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "audio",
                    "audio": audio_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    text = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        conversation,
        use_audio_in_video=False
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=False
    )

    if debug:
        print_input_debug(i, audio_path, transcript, conversation, text, audios, images, videos, inputs)

    input_len = inputs["input_ids"].shape[1]

    inputs = inputs.to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            use_audio_in_video=False,
            return_audio=False,
            max_new_tokens=10,
            do_sample=False
        )

    if isinstance(out_ids, tuple):
        out_ids = out_ids[0]

    full_text = processor.batch_decode(
        out_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    new_ids = out_ids[:, input_len:]

    new_text = processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    if debug:
        print("\nRaw decoded FULL text:")
        print(full_text)

        print("\nRaw decoded NEW text only:")
        print(new_text)

    if len(new_text.strip()) > 0:
        ans = new_text
    else:
        ans = full_text.split("Label:")[-1]

    pred = clean_pred(ans)

    if debug:
        print("Cleaned prediction:", pred)
        print("-" * 80)

    return pred

print("\nRunning debug predictions on first 5 samples only:")

debug_preds = []

for i in range(min(5, len(df))):
    audio_path = os.path.join(audio_dir, str(df.loc[i, audio_col]))
    transcript = str(df.loc[i, text_col])
    true_label = str(df.loc[i, label_col])

    pred = predict(audio_path, transcript, i=i, debug=True)

    debug_preds.append(pred)

    print("Sample:", i)
    print("True label:", true_label)
    print("Predicted label:", pred)

print("\nDebug first 5 predictions:", debug_preds)

print("\nNow running on full dataset...")

preds = []

for i, row in tqdm(df.iterrows(), total=len(df)):
    audio_path = os.path.join(audio_dir, str(row[audio_col]))
    transcript = str(row[text_col])

    pred = predict(audio_path, transcript, i=i, debug=False)
    preds.append(pred)

df["predicted_type"] = preds

print("\nPrediction counts:")
print(df["predicted_type"].value_counts())

print("\nFirst 20 predictions:")
print(df[[audio_col, text_col, label_col, "predicted_type"]].head(20))

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("qwen25_omni_speech_text_predictions_fixed.csv", index=False)

print("Saved predictions to qwen25_omni_speech_text_predictions_fixed.csv")

from google.colab import files
files.download("qwen25_omni_speech_text_predictions_fixed.csv")

CSV preview:
      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Columns: Index(['file_name', 'transcript', 'label'], dtype='object')
Audio column: file_name
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']
Missing audio files: 0

Checking first 3 audio-text pairs:

Sample 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
Transcript: Hello, Jane. I'm really glad to meet you here.
True label: declarative

Sample 1
Aud

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully.

Running debug predictions on first 5 samples only:

DEBUG SAMPLE: 0
Audio path: audio_files/0_0_d45.wav
Audio exists: True
Audio size bytes: 242594
Transcript: Hello, Jane. I'm really glad to meet you here.

Conversation content types:
Role: system
  type: text
Role: user
  type: audio
  type: text

Chat template first 600 characters:
["<|im_start|>system\nYou are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory inputs and generating text responses.<|im_end|>\n<|im_start|>user\n<|audio_bos|><|AUDIO|><|audio_eos|>You are a sentence type classifier.\n\nYou will receive both the speech audio and its transcript.\n\nClassify the sentence into exactly one of these labels:\n\ndeclarative: a normal statement or information-giving sentence.\ninterrogative: a question or sentence asking for information.\nimperative: a command, request, advice, instruction, or order.\nexclamatory: a sentence showing strong emotion, surpris


Raw decoded FULL text:
system
You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory inputs and generating text responses.
user
You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label

...


Raw decoded FULL text:
system
You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory inputs and generating text responses.
user
You are a sentence type classifier.

You will receive both the speech audio and its transcript.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Use both the audio and the transcript. The transcript gives the words, and the audio may give tone, emphasis, and intonation.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label

100%|██████████| 404/404 [2:41:45<00:00, 24.02s/it]


Prediction counts:
predicted_type
interrogative    135
declarative      112
exclamatory      106
imperative        51
Name: count, dtype: int64

First 20 predictions:
        file_name                                         transcript  \
0     0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1     0_0_d47.wav         I heard you and James are engaged at last.   
2    0_0_d215.wav                            Give me scotch, please.   
3     0_0_d49.wav            Hi, what brings you to my office today?   
4    0_0_d159.wav  Please help yourself at your dishes. I hope yo...   
5     0_0_d50.wav                                  Hello, Jack here.   
6    0_0_d403.wav            Excuse me. I wonder if you can help me.   
7     0_0_d63.wav            Umm, Jenny, are you having a good time?   
8    0_0_d681.wav  Excuse me. I bought this shirt yesterday, but ...   
9    0_0_d684.wav     Watch out! You're too close to the fire place.   
10   0_0_d704.wav                       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>